# Step 2: Main notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from method_b_catchment import extract_catchment_feature
from method_c_zonal import compute_zonal_stat, assign_max_buffer_value
from method_d_ratio import ratio_by_surface_type
from method_e_join import spatial_join_maxoverlap

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

## Imports

#### Import des segments

In [2]:
# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments, replace file path -->")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-1'
segments_gdf = gpd.read_file(f"{file_path}/step1_pedestrian_segments.gpkg")

Loading pedestrian segments, replace file path -->


#### Import du csv - méthode de traitement des attributs

In [3]:
# #Attributes info
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input'
# If the file attributes_info exists, load it
if os.path.exists(f"{file_path}/attributs_info.csv"):
    attributs_info = pd.read_csv(f"{file_path}/attributs_info.csv")
else:
    # If it does not exist, create a new DataFrame with the required structure
    attributs_info = pd.DataFrame(columns=['attribute', 'method', 'how', 'value_column', 'buffer_size'])
    
    # Example data to fill the DataFrame
    # You can modify this part to include the actual attributes you want to process
    attributs_info = pd.DataFrame({
        'attribute': ['arbre', 'accident', 'toilette', 'vitesse', 'eau'],
        'method': ['buffer', 'buffer', 'buffer', 'join', 'buffer'],
        'how' : ['count', 'sum', 'count', None, 'count'],  # 'how' can be 'count', 'mean', etc.
        'value_column': [None, 'PIETONS', None, 'TYPE_ZONE', None],  # Column to aggregate
        'buffer_size': [30, 50, 50, None, 100]  # Example buffer size for method A
    })
    attributs_info.to_csv(f"{file_path}/attributs_info.csv", index=False)

    # Add attributes later: too long to run
    # 'ratio_trottoir' 'ratio' None 'OBJET' '100'

#### Import des attributs et sauvegarde en gpkg (à modifier pour chaque attribut)

In [4]:
if not os.path.exists('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'):
    os.makedirs('/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs')
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'


#Chargement de la couhe des accidents 
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/accident/OTC_ACCIDENTS-SHP'
accident_gdf  = gpd.read_file(f"{file_path}/OTC_ACCIDENTS.shp")
print('Couche accident chargée avec succès')
accident_gdf = accident_gdf.to_crs(2056)
accident_gdf.to_file(f"{save_path}/accident.gpkg", driver='GPKG')

#Chargement de la couche des arbre
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/arbre/SIPV_ICA_ARBRE_ISOLE-SHP'
arbre_gdf = gpd.read_file(f"{file_path}/SIPV_ICA_ARBRE_ISOLE.shp")
print('Couche arbre chargée avec succès')
arbre_gdf = arbre_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
arbre_gdf.to_file(f"{save_path}/arbre.gpkg", driver='GPKG')

#Chargement de la couche des toilettes
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/toilette/VDG_WC_PUBLIC-SHP'
toilette_gdf = gpd.read_file(f"{file_path}/VDG_WC_PUBLIC.shp")
print('Couche toilette chargée avec succès')
toilette_gdf = toilette_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
toilette_gdf.to_file(f"{save_path}/toilette.gpkg", driver='GPKG')


#chargement de la couche des cours d'eau
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/eau/LCE_GRAPHE_EAU-SHP'
eau_gdf = gpd.read_file(f"{file_path}/LCE_GRAPHE_EAU.shp")
print('Couche eau chargée avec succès')
eau_gdf = eau_gdf.to_crs(2056)
eau_gdf = eau_gdf[
    eau_gdf["TYPE_SIEAU"].isin(["Cours principale", "Rive droite", "Rive gauche"]) &
    (eau_gdf["ETAT"] == "A ciel ouvert")
]
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
eau_gdf.to_file(f"{save_path}/eau.gpkg", driver='GPKG')

# Ajouter les autres couches d'attributs ici

Couche accident chargée avec succès
Couche arbre chargée avec succès
Couche toilette chargée avec succès
Couche eau chargée avec succès


In [5]:
#Chargement donaime routier
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/domaine_routier/CAD_DOMAINE_ROUTIER-SHP '
domaine_routier_gdf = gpd.read_file(f"{file_path}/CAD_DOMAINE_ROUTIER.shp")
print('Couche domaine routier chargée avec succès')
domaine_routier_gdf = domaine_routier_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
domaine_routier_gdf.to_file(f"{save_path}/domaine_routier.gpkg", driver='GPKG')

#Chargement de la couche des vitesse
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/vitesse/OTC_LIMITATIONS_VITESSE-SHP'
vitesse_gdf = gpd.read_file(f"{file_path}/OTC_LIMITATIONS_VITESSE.shp")
print('Couche vitesse chargée avec succès')
vitesse_gdf = vitesse_gdf.to_crs(2056)
save_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/output/step-2/gpkg_attributs'
vitesse_gdf.to_file(f"{save_path}/vitesse.gpkg", driver='GPKG')

Couche domaine routier chargée avec succès
Couche vitesse chargée avec succès


## Traitement des attributs

In [6]:
print("WARNING: This step can take a long time !! more than 12mn for ratios_trottoir")
#taking more than 10minutes to run ratios_trottoir treatment

In [7]:
# # Charger la table des attributs et méthodes
# attributs_info = pd.read_csv('../../Data/input/attributs_info.csv')

# # Charger les segments
# segments_gdf = gpd.read_file('../../Data/output/step-1/step1_pedestrian_segments.gpkg')

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segments_gdf
segments_gdf = segments_gdf[['osmid', 'maxspeed', 'geometry', 'segment_id']]

# Boucle sur chaque attribut
for _, row in attributs_info.iterrows():
    attribute_name = row['attribute']
    method = row['method']
    how = row['how']
    value_column = row['value_column']
    buffer_size = row['buffer_size']

    # Charger la couche attribut depuis gpd_attributs
    attribute_gdf = gpd.read_file(f"../../Data/output/step-2/gpkg_attributs/{attribute_name}.gpkg")
    attribute_gdf = attribute_gdf.to_crs(segments_gdf.crs)

    # Appliquer la méthode
    if method == "buffer":
        attribute_df = extract_buffer_feature(
            segments_gdf,
            attribute_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            how=how,
            value_column=value_column
        )
        print(f"Buffer feature extracted for {attribute_name} with method {method}")
    elif method == "catchment":
        attribute_df = extract_catchment_feature(
            segments_gdf=segments_gdf,
            feature_name=attribute_name,
            buffer_radius=buffer_size,
            points_gdf=attribute_gdf
        )
        print(f"Catchment feature extracted for {attribute_name} with method {method}")
    elif method == "zonal":
        attribute_df = compute_zonal_stat(
            segments_gdf,
            attribute_gdf,
            buffer_radius=buffer_size,
            value_column=value_column,
            stat=how
        )
        print(f"Zonal statistics computed for {attribute_name} with method {method}")

    elif method == "ratio":
        attribute_df = ratio_by_surface_type(
            segments_gdf=segments_gdf,
            surfaces_gdf=attribute_gdf,
            buffer_radius=buffer_size,
            surface_type_col=value_column
        )
        print(f"Ratio by surface type computed for {attribute_name} with method {method}")
    
    elif method == "join":
        attribute_df = spatial_join_maxoverlap(
        segments_gdf=segments_gdf,
        attribute_gdf=attribute_gdf,
        value_column=value_column,
        segment_id_col="segment_id",
        feature_name=attribute_name
        )
        print(f"Spatial join computed for {attribute_name} with method {method}") 
    
    # Ajoute d'autres méthodes si besoin

    # Ajouter la colonne au GeoDataFrame principal
    segments_gdf[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name]

# Sauvegarder le GeoDataFrame mis à jour
segments_gdf.to_csv('../../Data/output/step-2/step2_features.csv', index=False)
# Sauvegarder le GeoDataFrame mis à jour en format GPKG
segments_gdf.to_file('../../Data/output/step-2/step2_features.gpkg', driver='GPKG')

Buffer feature extracted for arbre with method buffer
Buffer feature extracted for accident with method buffer
Buffer feature extracted for toilette with method buffer
Spatial join computed for vitesse with method join
Buffer feature extracted for eau with method buffer


In [ ]:
pd.read_csv('../../Data/output/step-2/step2_features.csv').head(15)  # Afficher les 5 premières lignes pour vérification